# Calibration

https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html

In [4]:
import numpy as np
import cv2 as cv
import glob

# termination criteria
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
objp = np.zeros((11*13,3), np.float32)
objp[:,:2] = np.mgrid[0:13,0:11].T.reshape(-1,2)

# Arrays to store object points and image points from all the images.
objpoints = [] # 3d point in real world space
imgpoints = [] # 2d points in image plane.

images = glob.glob('../Data/calibration_img2/*.jpg')

for fname in images:
    # import img and transform it into an image in gray scale
    img = cv.imread(fname)
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

    # Find the chess board corners
    ret, corners = cv.findChessboardCorners(gray, (13,11), None)

    # If found, add object points, image points (after refining them)
    if ret == True:
        objpoints.append(objp)

        corners2 = cv.cornerSubPix(gray,corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners2)

        # Draw and display the corners
        cv.drawChessboardCorners(img, (13,11), corners2, ret)
        cv.imshow('img', img)
        cv.waitKey(500)

cv.destroyAllWindows()

In [5]:
ret, mtx, dist, rvecs, tvecs = cv.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

In [8]:
# Save parameters
np.save("mtx.npy", mtx)
np.save("dist.npy", dist)
np.savez("calib_para2/calib_params.npz", ret=ret, mtx=mtx, dist=dist, rvecs=rvecs, tvecs=tvecs)

# Undistortion

In [12]:
img = cv.imread("../Data/midbar_img/dimension_test/2025_06_04_1800.jpg")
h,  w = img.shape[:2]
newcameramtx, roi = cv.getOptimalNewCameraMatrix(mtx, dist, (w,h), 0, (w,h)) #  If the scaling parameter alpha=0, it returns undistorted image with minimum unwanted pixels.

In [13]:
# undistort
dst = cv.undistort(img, mtx, dist, None, newcameramtx)

# crop the image
x, y, w, h = roi
dst = dst[y:y+h, x:x+w]
cv.imwrite('Results/calib_test.png', dst)

True

# Validation

Values below 1 are considered acceptable for general tasks. For tasks that require precision, values between 0.2~0.5 are consiered ideal. 

Read the writing in the website below to undertand what the error implicates

https://www.mathworks.com/help/vision/ug/evaluating-the-accuracy-of-single-camera-calibration.html

In [14]:
mean_error = 0
for i in range(len(objpoints)):
    imgpoints2, _ = cv.projectPoints(objpoints[i], rvecs[i], tvecs[i], mtx, dist)
    error = cv.norm(imgpoints[i], imgpoints2, cv.NORM_L2)/len(imgpoints2)
    mean_error += error

print( "total error: {}".format(mean_error/len(objpoints)) )

total error: 0.6430835152029231
